[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/mrd_toy.ipynb)

# Manifold Relevance Determination with `gpc`

Multi-view sibling of Bayesian GPLVM ([CIP-0018](https://github.com/SheffieldML/GPc/blob/main/cip/cip0018.md)); native [mrd](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/mrd.md).

Requires `gpc>=0.1.0a8` (also on `0.1.0a7`).

```python
m = gpc.models.MRD([Y0, Y1], input_dim=3, num_inducing=10)
m.optimize()
fac = m.factorize_space(threshold=0.05)  # shared / private0 / private1
```

Shared \(q(\mathbf{X})\) and inducing \(Z\); per-view RBF-ARD so unused dims can shut off.


In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a8",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Two aligned synthetic views

Shared 2-D signal plus a private direction in each view.

In [ ]:
rng = np.random.default_rng(1)
N, Q = 40, 3
Xtrue = rng.normal(size=(N, Q))
# view 0 uses dims 0,1; view 1 uses dims 0,2
W0 = np.zeros((Q, 5)); W0[0, :] = rng.normal(size=5); W0[1, :] = rng.normal(size=5)
W1 = np.zeros((Q, 4)); W1[0, :] = rng.normal(size=4); W1[2, :] = rng.normal(size=4)
Y0 = Xtrue @ W0 + 0.05 * rng.normal(size=(N, 5))
Y1 = Xtrue @ W1 + 0.05 * rng.normal(size=(N, 4))
print(Y0.shape, Y1.shape)

## Fit MRD

In [ ]:
m = gpc.models.MRD([Y0, Y1], input_dim=3, num_inducing=8)
print(m)
m.optimize(iters=60, messages=False)
print("objective", m.objective(), "ELBO", m.elbo())
print("input scales view0", m.input_scales(0))
print("input scales view1", m.input_scales(1))
fac = m.factorize_space(threshold=0.05)
print("factorize_space", fac)
mu_x = m.latent_mean()

## Shared latent plane

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.scatter(mu_x[:, 0], mu_x[:, 1], c="C0", s=28, alpha=0.85)
Z = m.inducing_inputs()
ax.plot(Z[:, 0], Z[:, 1], "kx", ms=8, mew=1.5, label="inducing")
ax.set_xlabel("latent 0")
ax.set_ylabel("latent 1")
ax.set_title("MRD latent mean")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()